# Basic Usage

This notebook contains some basic examples with python code for accessing the TROVE API

It assumes you have the requests and pandas packages installed and the following environment variable defined

1. `TROVE_TOKEN = <your_api_token>` (from https://datatrove.as.arizona.edu/users/profile/)

## Imports & Useful Variables

In [9]:
import os
import requests
import pandas as pd
from time import strftime

trove_url = "https://trove.as.arizona.edu/api/"
token = os.environ.get("TROVE_TOKEN")
nle_name = "GW190814" # a nonlocalized event (e.g., GW event) name

## Getting Scores

The primary thing users want to do is download the scores via the API to help them prioritize follow-up. To get all scores of candidates associated with a transient we can use the "score" endpoint: 

In [10]:
response = requests.get(
    f"{trove_url}/score/{nle_name}",
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

scores = response.json()

scores[0]

{'id': 3075,
 'target': {'id': 41709156,
  'name': 'AT2019nyw',
  'ra': 12.154324533074,
  'dec': -23.980024160676,
  'tns_redshift': None,
  'tns_classification': None},
 'nonlocalizedevent': {'id': 21085,
  'event_id': 'GW190814',
  'event_type': 'GW'},
 'score': {'KN': 0.6396198048370905}}

Which is the highest scoring candidate. The dictionary has a few important keys

1. The candidate "id". This will be unique for an NLE
2. `target`: A dictionary of target information including the TNS (or closest) name, ra, dec, tns_redshift, tns_classification
3. `nonlocalizedevent`: A dictionary with information about the NLE
4. `score`: A dictionary with keys for different scoring algorithms. For NS bearing GW events this will always have the key "KN". For SSM events it may also have keys for "KN-in-SN" and "super-KN" vetting algorithms.

But, we can also get the lowest scoring candidate:

In [11]:
scores[-1]

{'id': 3089,
 'target': {'id': 41709244,
  'name': 'AT2019nuy',
  'ra': 12.7099877991,
  'dec': -25.4915413351,
  'tns_redshift': None,
  'tns_classification': None},
 'nonlocalizedevent': {'id': 21085,
  'event_id': 'GW190814',
  'event_type': 'GW'},
 'score': {'KN': 0.0}}

Or, the top ten scores:

In [12]:
scores[:10]

[{'id': 3075,
  'target': {'id': 41709156,
   'name': 'AT2019nyw',
   'ra': 12.154324533074,
   'dec': -23.980024160676,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.6396198048370905}},
 {'id': 3082,
  'target': {'id': 41709262,
   'name': 'AT2019nuz',
   'ra': 12.4677684328,
   'dec': -25.5237821721,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.568923428074759}},
 {'id': 3078,
  'target': {'id': 41709427,
   'name': 'AT2019noq',
   'ra': 12.1995075422,
   'dec': -25.3065162788,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.4280890575048045}},
 {'id': 3071,
  'target': {'id': 41709122,
   'name': 'AT2019oae',
   'ra': 12.318312693987

If we instead only want the score of a specific transient we can use some query parameters in the URL.

In [13]:
response = requests.get(
    f"{trove_url}/score/{nle_name}?candidate_names=AT2019pck",
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

at2019pck_score = response.json()
at2019pck_score

[{'id': 3064,
  'target': {'id': 41708423,
   'name': 'AT2019pck',
   'ra': 11.851431586385,
   'dec': -25.226056630082,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 2.1622736642161166e-13}}]

Or, say we are interested in a few different targets

In [14]:
response = requests.get(
    f"{trove_url}/score/{nle_name}?candidate_names=AT2019pck,AT2019odc",
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

scores = response.json()
scores

[{'id': 3036,
  'target': {'id': 41709056,
   'name': 'AT2019odc',
   'ra': 11.507039,
   'dec': -25.45915,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 0.025883295138173296}},
 {'id': 3064,
  'target': {'id': 41708423,
   'name': 'AT2019pck',
   'ra': 11.851431586385,
   'dec': -25.226056630082,
   'tns_redshift': None,
   'tns_classification': None},
  'nonlocalizedevent': {'id': 21085,
   'event_id': 'GW190814',
   'event_type': 'GW'},
  'score': {'KN': 2.1622736642161166e-13}}]

## Getting Scores with a Cone Search

Say you are trying to plan an observing follow-up campaign and you want to prioritize the top 10 targets that are visible to your telescope. To do this, you can get a ranked list of candidates with a cone search:

In [15]:
ra_ctr = 13.163209
dec_ctr = -25.004159
radius = 3600 # 5 arcmin

response = requests.get(
    f"{trove_url}/score/{nle_name}/cone_search?ra={ra_ctr}&dec={dec_ctr}&radius={radius}",
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

scores = response.json()
len(scores)

23

## Uploading New Targets & Photometry

Finally, another common use case is reporting new targets and/or photometry that you got! For this, we can use the `target` endpoint.

Please **do not** change the RA, Dec in the example below! 

In [17]:
upload_url = f"{trove_url}/target/upload"
now_yymmdd_hhmmss = strftime("%y%m%d_%H%M%S") # timestamp, for easy cleanup of test candidates later

target_data = [
    {
        "name": f"test_{now_yymmdd_hhmmss}",
        "ra": 0,
        "dec": 0
    }
]

response = requests.post(
    upload_url,
    json=target_data,
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

print(response.status_code)

200


Or, if you want to also provide photometry

In [9]:
target_data = [
  {
    "name": f"test_{now_yymmdd_hhmmss}",
    "ra": 0,
    "dec": 0,
    "photometry": [
      {
        "jd": 2461134.3933751667,
        "telescope": "TEST",
        "filter": "r",
        "magnitude": 20,
        "error": 0.1
      },
      {
        "jd": 2461133.3933751667,
        "telescope": "TEST",
        "filter": "r",
        "limit": 23.1
      }
    ]
  }
]

response = requests.post(
    upload_url,
    json=target_data,
    headers={"accept": "*/*", "Authorization": f"Bearer {token}"}
)

print(response.status_code)

200
